# 01. Tìm hiểu dữ liệu

Notebook này xác định cấu trúc tám bảng Home Credit, đơn vị quan sát, khóa liên kết và ý nghĩa của `TARGET`. Cuối notebook, nhóm cố định tập train, validation, test và các CV fold để những bước sau dùng cùng một thiết kế.

Tỷ lệ chia 60/20/20 và seed 42 là lựa chọn của nhóm. `TARGET` được diễn giải theo định nghĩa trong từ điển Home Credit.



## Mục lục

- [01. Kết nối và đọc dữ liệu nguồn từ MySQL](#part-1)
- [02. Giới thiệu các bảng dữ liệu](#part-2)
- [03. Tổng quan cấu trúc dữ liệu](#part-3)
- [04. Từ điển dữ liệu và biến mục tiêu](#part-4)
- [05. Đơn vị quan sát và khóa](#part-5)
- [06. Quan hệ giữa các bảng](#part-6)
- [07. Phân chia dữ liệu](#part-7)
- [08. Tổng kết và chuyển sang notebook 02](#part-8)

<a id="muc-1"></a>

<a id="part-1"></a>

## 01. Kết nối và đọc dữ liệu nguồn từ MySQL

### 01.1. Thư viện và đường dẫn
dùng chung lớp `Connector` và `HomeCreditData` cho việc đọc dữ liệu
và kiểm tra cấu trúc. CSDL đã có tám bảng `raw_*` trước khi chạy notebook;
cách chuẩn bị được mô tả trong [hướng dẫn MySQL](../sql/README.md).

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from connectors.Connector import Connector
from models.HomeCreditData import HomeCreditData

metadata_dir = project_dir / 'data/metadata'
split_dir = project_dir / 'data/splits'
report_dir = project_dir / 'reports/data_understanding'
split_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)


`connectors/Connector.py` phụ trách kết nối và thực thi SQL; `models/HomeCreditData.py` lưu các bảng đã đọc và cung cấp thao tác dùng chung. Các mục dưới đây trình bày cách notebook lấy dữ liệu và kết quả đối soát.


### 01.2. Kết nối MySQL và đọc cấu trúc bảng
kết nối đúng CSDL và lấy tên cột, kiểu dữ liệu của đủ tám bảng.
Mật khẩu được nhập kín hoặc đọc từ biến môi trường, không ghi vào notebook.

In [2]:
connector = Connector()
connector.connect()


credit_data = HomeCreditData(connector)
column_overview = credit_data.loadColumns()
print('Đã kết nối CSDL:', connector.database)


Các bảng nguồn được đọc qua đối tượng `credit_data`, sử dụng chung
kết nối `connector`. Chưa thay đổi giá trị hoặc làm sạch dữ liệu nguồn.

### 01.3. Đọc hai bảng hồ sơ hiện tại
SELECT hai bảng `raw_application_train` và `raw_application_test`
thành DataFrame, giữ nguyên thứ tự nguồn để tái lập cách chia dữ liệu đã chốt.

`loadApplications()` giữ kết quả trong đối tượng `credit_data`. Các cell sau
dùng lại `application` và `scoring`; gọi lại phương thức này trên cùng đối tượng
không truy vấn lại. Muốn cập nhật dữ liệu nguồn thì tạo đối tượng mới và nạp lại.
Không tạo hoặc đọc cache `data/raw_mysql` trong luồng phân tích notebook này.

In [3]:
# Đọc một lần; các bước bên dưới dùng lại hai DataFrame này.
application, scoring = credit_data.loadApplications()

print('Application train:', application.shape)
print('Application test:', scoring.shape)
application.head()


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,...,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,100006,0,Cash loans,F,N,Y,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


### Kết luận mục 01

Hai bảng hồ sơ được đọc trực tiếp từ MySQL thành DataFrame và giữ trong `credit_data` để dùng lại. `application_train` có 307.511 hồ sơ, còn `application_test` có 48.744 hồ sơ và không có `TARGET`; notebook chưa làm sạch hoặc thay đổi dữ liệu nguồn.


<a id="muc-2"></a>

<a id="part-2"></a>

## 02. Giới thiệu các bảng dữ liệu

Xem năm dòng mẫu của mỗi bảng để xác định vai trò và đơn vị quan sát.
Hai bảng application dùng DataFrame đã nạp; bảng lịch sử lấy mẫu bằng SQL `LIMIT 5`.

In [4]:
preview_application = credit_data.getPreview('application_train')
preview_application


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,...,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,100006,0,Cash loans,F,N,Y,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,...,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [5]:
preview_bureau = credit_data.getPreview('bureau')
preview_bureau


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,...,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0000,...,"91,323.0000",0.0000,NaN,0.0000,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,"1,075.0000",...,"225,000.0000","171,342.0000",NaN,0.0000,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0000,...,"464,323.5000",NaN,NaN,0.0000,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,...,"90,000.0000",NaN,NaN,0.0000,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,"1,197.0000",...,"2,700,000.0000",NaN,NaN,0.0000,Consumer credit,-21,NaN


In [ ]:
preview_payments = credit_data.getPreview('installments_payments')
preview_payments

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0000,6,"-1,180.0000","-1,187.0000","6,948.3600","6,948.3600"
1,1330831,151639,0.0000,34,"-2,156.0000","-2,156.0000","1,716.5250","1,716.5250"
2,2085231,193053,2.0000,1,-63.0000,-63.0000,"25,425.0000","25,425.0000"
3,2452527,199697,1.0000,3,"-2,418.0000","-2,426.0000","24,350.1300","24,350.1300"
4,2714724,167756,1.0000,2,"-1,383.0000","-1,366.0000","2,165.0400","2,160.5850"


### Kết luận mục 02

Hai bảng application mô tả hồ sơ hiện tại; các bảng còn lại chứa lịch sử từ bureau hoặc các khoản vay trước tại Home Credit. Mỗi nguồn có đơn vị quan sát riêng, nên cần xác định đúng khóa và mức tổng hợp trước khi ghép.


<a id="muc-3"></a>

<a id="part-3"></a>

## 03. Tổng quan cấu trúc dữ liệu

Xác định chính xác số dòng, số cột và nhóm kiểu dữ liệu của từng bảng.
Số dòng lấy bằng SQL `COUNT(*)`; tên cột và kiểu lấy từ `information_schema.COLUMNS`.
Không dùng số dòng ước lượng của MySQL hoặc metadata Parquet để thay cho kiểm đếm.
Dung lượng file nén không được đưa vào bảng này vì không phải quy mô lưu trữ MySQL.

In [7]:
dataset_overview = credit_data.processOverview()
dataset_overview


,table,rows,columns
0,application_train,307511,122
1,application_test,48744,121
2,bureau,1716428,17
3,bureau_balance,27299925,3
4,previous_application,1670214,37
5,pos_cash_balance,10001358,8
6,installments_payments,13605401,8
7,credit_card_balance,3840312,23


In [8]:
dtype_counts = column_overview.groupby(['table', 'dtype']).size()
dtype_summary = dtype_counts.unstack(fill_value=0)
dtype_summary


dtype,bigint,double,text
table,,,
application_test,40,65,16
application_train,41,65,16
bureau,6,8,3
bureau_balance,2,0,1
credit_card_balance,7,15,1
installments_payments,3,5,0
pos_cash_balance,5,2,1
previous_application,6,15,16


In [9]:
id_columns = column_overview[
    column_overview["column"].str.startswith("SK_ID")
]
id_columns


,table,column,dtype
0,application_test,SK_ID_CURR,bigint
121,application_train,SK_ID_CURR,bigint
243,bureau,SK_ID_CURR,bigint
244,bureau,SK_ID_BUREAU,bigint
260,bureau_balance,SK_ID_BUREAU,bigint
263,credit_card_balance,SK_ID_PREV,bigint
264,credit_card_balance,SK_ID_CURR,bigint
286,installments_payments,SK_ID_PREV,bigint
287,installments_payments,SK_ID_CURR,bigint
294,pos_cash_balance,SK_ID_PREV,bigint


### Kết luận mục 03

Tám bảng có quy mô và cấu trúc khác nhau; `bureau_balance` là bảng lớn nhất theo số dòng. Các mã định danh được giữ để nối và truy vết, không xem là biến dự báo chỉ vì chúng có kiểu số.


<a id="muc-4"></a>

<a id="part-4"></a>

## 04. Từ điển dữ liệu và biến mục tiêu

Đối chiếu tên cột với tài liệu gốc và xác định đúng ý nghĩa của `TARGET`.

Trong từ điển, một số tên có khoảng trắng và `SK_ID_BUREAU` được ghi là
`SK_BUREAU_ID`. Việc chuẩn hóa dưới đây chỉ phục vụ tra cứu metadata, không sửa dữ liệu nguồn.

In [10]:
dictionary = pd.read_csv(
    metadata_dir / "HomeCredit_columns_description.csv",
    encoding="latin1"
)

dictionary["table_lookup"] = dictionary["Table"].str.strip().str.lower()
dictionary["column_lookup"] = dictionary["Row"].str.strip()
dictionary["column_lookup"] = dictionary["column_lookup"].replace({
    "SK_BUREAU_ID": "SK_ID_BUREAU"
})

column_dictionary = column_overview.copy()
column_dictionary["table_lookup"] = column_dictionary["table"] + ".csv"
is_application = column_dictionary["table"].isin(["application_train", "application_test"])
column_dictionary.loc[
    is_application,
    "table_lookup"
] = "application_{train|test}.csv"

column_dictionary = column_dictionary.merge(
    dictionary[["table_lookup", "column_lookup", "Row", "Description", "Special"]],
    left_on=["table_lookup", "column"],
    right_on=["table_lookup", "column_lookup"],
    how="left",
    validate="many_to_one"
)


column_dictionary[["table", "column", "Description"]].head(12)


,table,column,Description
0,application_test,SK_ID_CURR,ID of loan in our sample
1,application_test,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving
2,application_test,CODE_GENDER,Gender of the client
3,application_test,FLAG_OWN_CAR,Flag if the client owns a car
4,application_test,FLAG_OWN_REALTY,Flag if client owns a house or flat
5,application_test,CNT_CHILDREN,Number of children the client has
6,application_test,AMT_INCOME_TOTAL,Income of the client
7,application_test,AMT_CREDIT,Credit amount of the loan
8,application_test,AMT_ANNUITY,Loan annuity
9,application_test,AMT_GOODS_PRICE,For consumer loans it is the price of the goods for which the loan is given


In [11]:
dictionary.loc[
    (dictionary["column_lookup"] == "TARGET"),
    ["Row", "Description"]
]


,Row,Description
1,TARGET,Target variable (1 - client with payment difficulties: he/she had late payment more than X days ...


In [12]:
application_labels = application[['SK_ID_CURR', 'TARGET']].copy()
scoring_ids = scoring[['SK_ID_CURR']].copy()

target_counts = application_labels['TARGET'].value_counts().sort_index()
target_summary = target_counts.rename_axis('TARGET').reset_index(name='rows')
target_summary['percentage'] = 100 * target_summary['rows'] / len(application_labels)
target_summary


,TARGET,rows,percentage
0,0,282686,91.9271
1,1,24825,8.0729


Từ điển đã được đối chiếu với 339 cặp bảng/cột. Trong 307.511 hồ sơ có nhãn, 24.825 hồ sơ thuộc lớp 1 (khoảng **8,07%**) và lớp 0 chiếm khoảng **91,93%**, nên dữ liệu mất cân bằng.

`TARGET = 1` biểu thị khó khăn thanh toán theo định nghĩa Home Credit: khách hàng trễ hơn X ngày ở ít nhất một trong Y kỳ đầu. Tài liệu được dùng ở đây không nêu cụ thể X và Y; không diễn giải nhãn này thành nợ xấu theo quy định hay vỡ nợ trong 12 tháng. `application_test` là tập nguồn không có nhãn, khác với test nội bộ được tạo ở mục 07.


### 04.1. Ý nghĩa nghiệp vụ của các nhóm thông tin

Bảng dưới đây gom các nguồn dữ liệu theo vai trò trong phân tích rủi ro thanh toán. Đây là cách nhóm tổ chức EDA, không phải phân loại bắt buộc trong sách.

| Nhóm thông tin | Biến hoặc nguồn đại diện | Ý nghĩa khi phân tích | Giới hạn diễn giải |
|---|---|---|---|
| Thu nhập và nghĩa vụ thanh toán | AMT_INCOME_TOTAL, AMT_ANNUITY, AMT_CREDIT | Quy mô thu nhập, khoản vay và nghĩa vụ trả theo thông tin hồ sơ | Chưa xác minh kỳ đo và tổng nghĩa vụ; chưa gọi annuity/income là DTI chuẩn hoặc AMT_CREDIT là EAD |
| Cá nhân và tính ổn định | DAYS_BIRTH, DAYS_EMPLOYED, NAME_INCOME_TYPE, DAYS_REGISTRATION | Tuổi, thời điểm bắt đầu việc làm, loại thu nhập và thay đổi đăng ký | DAYS là số ngày tương đối; tách sentinel trước khi diễn giải thâm niên; không suy ra nguyên nhân rủi ro từ nghề nghiệp/tuổi |
| Lịch sử tín dụng và thanh toán | bureau/bureau_balance, previous_application, POS, credit card, installments | Khoản tín dụng trước, dư nợ/quá hạn, snapshot tháng và các lần thanh toán | Tổng hợp đúng đơn vị khoản/tháng/kỳ lên SK_ID_CURR; chỉ dùng thông tin sẵn có lúc đánh giá; không có lịch sử không có nghĩa đã trả nợ tốt |
| Thông tin sở hữu | FLAG_OWN_CAR, OWN_CAR_AGE, FLAG_OWN_REALTY | Thông tin tham khảo về sở hữu tài sản và sự ổn định | Sở hữu chưa chứng minh tài sản được thế chấp; tuổi xe có thể không áp dụng khi không sở hữu xe |
| Điểm từ nguồn bên ngoài | EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3 | Các điểm đã chuẩn hóa theo từ điển Home Credit; kiểm tra liên hệ thực tế với TARGET | Chưa biết nhà cung cấp, công thức và thời hạn dự báo; không tự gọi là FICO hoặc PD |

Tên biến và phạm vi trường lấy từ từ điển Home Credit; cách gom nhóm trong bảng là lựa chọn phân tích của nhóm.

### Kết luận mục 04

Năm nhóm thông tin này sẽ dẫn dắt EDA và lựa chọn biến ở các bước sau. Trong sách, *Capacity* của 6 Cs nói về năng lực/thẩm quyền pháp lý; khả năng tạo tiền trả nợ thuộc *Cash*. Vì vậy, không gán mọi cột trong dữ liệu vào đủ sáu Cs.


<a id="muc-5"></a>

<a id="part-5"></a>

## 05. Đơn vị quan sát và khóa

Kiểm tra khóa ứng viên và xác định bảng nào có nhiều dòng cho cùng hồ sơ.

| Bảng | Một dòng đại diện cho | Khóa kiểm tra |
|---|---|---|
| application_train/test | Hồ sơ vay hiện tại | SK_ID_CURR |
| bureau | Khoản tín dụng tại credit bureau | SK_ID_BUREAU |
| bureau_balance | Trạng thái một khoản bureau trong một tháng | SK_ID_BUREAU, MONTHS_BALANCE |
| previous_application | Hồ sơ vay trước | SK_ID_PREV |
| pos_cash_balance / credit_card_balance | Snapshot tháng của một khoản trước | SK_ID_PREV, MONTHS_BALANCE |
| installments_payments | Bản ghi thanh toán | SK_ID_PREV, phiên bản, số kỳ chỉ xác định **kỳ**, chưa chắc xác định duy nhất một bản ghi |

Các phép kiểm tra dùng `HomeCreditData.processKeyChecks()`. Với bảng lịch sử lớn, chỉ đọc các cột khóa và chủ hồ sơ cần thiết; trên cùng DataFrame kiểm tra cả khóa lẫn quan hệ rồi giải phóng. Kết quả được giữ để mục 6 dùng lại.

In [13]:
candidate_keys = {
    "application_train": ["SK_ID_CURR"],
    "application_test": ["SK_ID_CURR"],
    "bureau": ["SK_ID_BUREAU"],
    "bureau_balance": ["SK_ID_BUREAU", "MONTHS_BALANCE"],
    "previous_application": ["SK_ID_PREV"],
    "pos_cash_balance": ["SK_ID_PREV", "MONTHS_BALANCE"],
    "credit_card_balance": ["SK_ID_PREV", "MONTHS_BALANCE"],
    "installments_payments": ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]
}


key_checks = credit_data.processKeyChecks(candidate_keys)
key_checks


,table,key,rows,missing_key_rows,duplicate_key_rows,expected_unique
0,application_train,SK_ID_CURR,307511,0,0,True
1,application_test,SK_ID_CURR,48744,0,0,True
2,bureau,SK_ID_BUREAU,1716428,0,0,True
3,bureau_balance,"SK_ID_BUREAU, MONTHS_BALANCE",27299925,0,0,True
4,previous_application,SK_ID_PREV,1670214,0,0,True
5,pos_cash_balance,"SK_ID_PREV, MONTHS_BALANCE",10001358,0,0,True
6,credit_card_balance,"SK_ID_PREV, MONTHS_BALANCE",3840312,0,0,True
7,installments_payments,"SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER",13605401,0,653483,False


### Kết luận mục 05

Các khóa bắt buộc không thiếu và không lặp trong dữ liệu đã kiểm tra. Riêng khóa kỳ của `installments_payments` có **653.483 dòng lặp dư**; một kỳ có thể gồm nhiều lần thanh toán nên chưa có cơ sở xóa các dòng này như bản ghi trùng. Notebook 02 tiếp tục kiểm tra chúng theo đơn vị quan sát.


<a id="muc-6"></a>

<a id="part-6"></a>

## 06. Quan hệ giữa các bảng

Kiểm tra khả năng liên kết, trường hợp không có bảng cha và độ bao phủ lịch sử.

```text
application_train / application_test  [SK_ID_CURR]
    ├── bureau  [SK_ID_BUREAU] ── bureau_balance
    └── previous_application  [SK_ID_PREV]
            ├── pos_cash_balance
            ├── credit_card_balance
            └── installments_payments
```

Ba bảng dưới cùng còn có `SK_ID_CURR` trực tiếp. Cần phân biệt không tìm thấy
`SK_ID_PREV` ở bảng cha với không xác định được hồ sơ hiện tại.

Phần này kiểm tra liên kết trên toàn nguồn, không dùng TARGET để phân tích.
Chỉ các cột khóa/chủ hồ sơ của một bảng lớn được ghép thành DataFrame để kiểm tra; không nạp toàn bộ cột của cả sáu bảng lịch sử vào RAM. DataFrame được giải phóng sau mỗi bảng, chỉ giữ các kết quả kiểm tra.
**Cách dùng dữ liệu:** lấy kết quả liên kết đã tính ở mục 5 bằng `processRelationships()`, không SELECT lại từng bảng lịch sử.

In [14]:
relationships = credit_data.processRelationships()
relationships


,table,missing_parent_rows,owner_conflict_rows,unresolved_owner_rows,outside_scope_rows,applications_with_history,total_applications,coverage_pct
0,bureau,0,0,0,0,305811,356255,85.8405
1,bureau_balance,3120184,0,3120184,0,134542,356255,37.7656
2,previous_application,0,0,0,0,338857,356255,95.1164
3,pos_cash_balance,340561,0,0,0,337252,356255,94.6659
4,credit_card_balance,1082816,0,0,0,103558,356255,29.0685
5,installments_payments,1250826,0,0,0,339587,356255,95.3213


### Kết luận mục 06

Các bảng lịch sử nối được theo những khóa đã xác định, nhưng một số dòng không tìm thấy bảng cha. `bureau_balance` có **3.120.184 dòng** chưa nối được với `bureau`, nên không thể xác định hồ sơ hiện tại từ dữ liệu hiện có. Các dòng này không tự động bị loại; lịch sử một-nhiều cũng không nên ghép thẳng vào hồ sơ nếu chưa tổng hợp đúng cấp.


<a id="muc-7"></a>

<a id="part-7"></a>

## 07. Phân chia dữ liệu

Cố định ID train, validation, test và ba CV fold để các notebook sau
sử dụng cùng một cách chia.

Thiết kế đã chọn là 60% train, 20% validation và 20% test nội bộ. Tách test trước,
sau đó lấy 25% phần còn lại làm validation. Cả hai lần phân tầng theo TARGET và dùng seed 42.

Train dùng phân tích và huấn luyện; validation chọn ngưỡng; test đánh giá cuối.
Notebook cũ đã từng mô tả toàn nguồn, nên không xem test là chưa từng được quan sát.
Từ bước này, EDA có nhãn và các quyết định đặc trưng chỉ sử dụng train.

In [15]:
row_indices = np.arange(len(application_labels))

remaining_indices, test_indices = train_test_split(
    row_indices,
    test_size=0.20,
    stratify=application_labels["TARGET"],
    random_state=42
)

train_indices, validation_indices = train_test_split(
    remaining_indices,
    test_size=0.25,
    stratify=application_labels.iloc[remaining_indices]["TARGET"],
    random_state=42
)

split_ids = application_labels[["SK_ID_CURR"]].copy()
split_ids["split"] = ""
split_ids.loc[train_indices, "split"] = "train"
split_ids.loc[validation_indices, "split"] = "validation"
split_ids.loc[test_indices, "split"] = "test"
split_ids["cv_fold"] = -1


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
train_target = application_labels.iloc[train_indices]["TARGET"]

for fold, (_, fold_indices) in enumerate(cv.split(np.zeros(len(train_indices)), train_target)):
    split_ids.loc[train_indices[fold_indices], "cv_fold"] = fold

split_ids = split_ids.sort_values("SK_ID_CURR").reset_index(drop=True)


split_ids.head()

,SK_ID_CURR,split,cv_fold
0,100002,train,2
1,100003,train,2
2,100004,train,0
3,100006,validation,-1
4,100007,validation,-1


In [17]:
split_labels = split_ids.merge(
    application_labels, on='SK_ID_CURR', validate='one_to_one'
)

split_summary = split_labels.groupby('split')['TARGET'].agg(
    rows='size', positive_count='sum', positive_rate='mean'
)
split_summary = split_summary.reset_index()
split_summary['negative_count'] = split_summary['rows'] - split_summary['positive_count']
split_summary


,split,rows,positive_count,positive_rate,negative_count
0,test,61503,4965,0.0807,56538
1,train,184506,14895,0.0807,169611
2,validation,61502,4965,0.0807,56537


In [ ]:
train_mask = split_labels['split'] == 'train'
train_split = split_labels.loc[train_mask]
fold_summary = train_split.groupby('cv_fold')['TARGET'].agg(
    rows='size', positive_count='sum', positive_rate='mean'
)
fold_summary = fold_summary.reset_index()
fold_summary

,cv_fold,rows,positive_count,positive_rate
0,0,61502,4965,0.0807
1,1,61502,4965,0.0807
2,2,61502,4965,0.0807


**Lưu ID và fold cho các notebook sau**

In [ ]:
# Không ghi đè cách chia đã dùng trong các thực nghiệm trước.
split_file = split_dir / 'application_split.parquet'
if split_file.exists():
    saved_split = pd.read_parquet(split_file)
    if not split_ids.equals(saved_split):
        raise ValueError('ID hoặc fold khác bản đã chốt. Dừng trước khi ghi dữ liệu.')
else:
    split_ids.to_parquet(split_file, index=False)

saved_rows = credit_data.saveSplit(split_ids)
print('Phân tập đã lưu:', saved_rows, 'hồ sơ')

**Lưu các bảng dùng để đối chiếu ở notebook 02**

In [ ]:
key_checks.to_csv(report_dir / 'key_checks.csv', index=False)
relationships.to_csv(report_dir / 'relationships.csv', index=False)
connector.disConnect()

### Kết luận mục 07

Tập train có **184.506** hồ sơ, validation **61.502**, test nội bộ **61.503**; tỷ lệ TARGET dương ở mỗi tập xấp xỉ **8,07%**. ID được lưu cùng ba CV fold chỉ tạo trên train. Các notebook sau dùng split này thay vì chia lại dữ liệu.


<a id="muc-8"></a>

<a id="part-8"></a>

## 08. Tổng kết và chuyển sang notebook 02

Notebook này chốt cách hiểu dữ liệu trước khi kiểm tra chất lượng: cấu trúc bảng, TARGET, khóa, quan hệ giữa các nguồn và thiết kế chia tập.

### Kết luận

- Đã xác định vai trò, quy mô và nhóm kiểu dữ liệu của tám bảng.
- Đã đối chiếu từ điển, làm rõ giới hạn diễn giải TARGET và ghi nhận lớp dương chiếm khoảng 8,07%.
- Đã kiểm tra khóa, quan hệ giữa bảng và các dòng lịch sử thiếu bảng cha.
- Đã lưu ID train, validation, test và CV fold theo thiết kế đã chọn.

Bước tiếp theo là kiểm tra dữ liệu thiếu, sentinel, giá trị cực đoan, thời điểm lịch sử và nhiều lần thanh toán trong một kỳ. Notebook này chưa xử lý các vấn đề đó và không thay đổi dữ liệu nguồn.

Các bảng kết quả được lưu ở `reports/data_understanding/`.

Các cách diễn giải nghiệp vụ được giới hạn theo dữ liệu thực sự có trong bộ nguồn.
